# Analyse van `data/raw/`
Analyse van de ruwe parquet-bestanden met DuckDB, vóórdat er staging-modellen
op gebouwd worden. Doel is drie vragen beantwoorden per bestand:

1. **Omvang** — hoeveel rijen, kolommen, bytes?
2. **Grain** — wat is de sleutel, en is die uniek?
3. **Vulling** — welke kolommen zijn leeg, welke types zitten er in, en hoeveel is uniek?

In [102]:
import os
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = (ROOT / os.environ.get("DBT_RAW_DIR", "data/raw")).resolve()
GLOB = f"{RAW}/*.parquet"

con = duckdb.connect()  # in-memory; de parquet-bestanden worden niet gekopieerd

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)


def q(sql: str, **params) -> pd.DataFrame:
    """Query uitvoeren en als DataFrame teruggeven.

    Alleen `.format()` als er params meegegeven zijn: verschillende queries
    hieronder bevatten een letterlijke `{4}` in een regex, en die mag niet als
    placeholder gelezen worden.
    """
    return con.sql(sql.format(**params) if params else sql).df()


def bron(naam: str) -> str:
    """Bestandsnaam -> quoted pad, zodat het als tabel in een FROM kan."""
    return f"'{RAW}/{naam}.parquet'"


print(RAW, "—", len(list(RAW.glob("*.parquet"))), "parquet-bestanden")

/Users/ruudjuffermans/Projects/datavakwerk/open-data-warehouse/data/raw — 23 parquet-bestanden


## Tabellen

`parquet_file_metadata()` leest alleen de footer: rijaantallen en bestandsgrootte
zonder de kolommen te scannen. Het aantal kolommen komt uit `parquet_schema()`
(de root-node telt niet mee, vandaar de filter op `num_children is null`).

In [59]:
inventaris = q("""
    with meta as (
        select
            regexp_replace(file_name, '.*/', '') as bestand,
            num_rows,
            num_row_groups,
            file_size_bytes
        from parquet_file_metadata('{glob}')
    ),
    kolommen as (
        select
            regexp_replace(file_name, '.*/', '') as bestand,
            count(*) as n_kolommen
        from parquet_schema('{glob}')
        where num_children is null
        group by 1
    )
    select
        meta.bestand,
        meta.num_rows                                as rijen,
        kolommen.n_kolommen                          as kolommen,
        round(meta.file_size_bytes / 1024.0 / 1024, 2) as mb,
        meta.num_row_groups                          as rowgroups,
        round(meta.file_size_bytes / nullif(meta.num_rows, 0), 1) as bytes_per_rij
    from meta
    join kolommen using (bestand)
    order by meta.num_rows desc
""", glob=GLOB)

inventaris

,bestand,rijen,kolommen,mb,rowgroups,bytes_per_rij
0,rdw_geconstateerde_gebreken.parquet,1333714,8,12.07,27,9.5
1,rdw_gekentekende_voertuigen.parquet,500000,98,25.12,10,52.7
2,rdw_brandstof.parquet,443167,36,4.40,9,10.4
3,cbs_70072ned_motorvoertuigen_gemeente.parquet,213101,7,1.18,5,5.8
4,cbs_85237ned_personenautos_brandstof.parquet,80592,7,0.34,2,4.5
5,cbs_gebieden_2015.parquet,22401,6,0.09,1,4.2
6,cbs_gebieden_2018.parquet,21660,6,0.09,1,4.2
7,cbs_gebieden_2016.parquet,20670,6,0.08,1,4.2
8,cbs_gebieden_2023.parquet,19494,6,0.08,1,4.3
9,cbs_gebieden_2024.parquet,19494,6,0.08,1,4.3


## 3. Kolommen

`SUMMARIZE` doet per kolom min/max, `approx_unique` (HyperLogLog) en
`null_percentage` in één scan. Op het grootste bestand (500k × 98) is dat ~1 s.

`leeg_percentage` telt daarnaast lege strings apart: die zijn in parquet niet NULL,
maar betekenen hier hetzelfde. Het verschil tussen die twee kolommen is precies wat
een `nullif(x, '')` in staging zou opruimen.

In [71]:
tabellen = q("""
        select
            regexp_replace(file_name, '.*/', '') as bestand,
        from parquet_file_metadata('{glob}')
""", glob=GLOB)

for bestand in tabellen.bestand:
    print(bestand.strip(".parquet"))

cbs_70072ned_measurecodes
cbs_70072ned_motorvoertuigen_gemeen
cbs_70072ned_periodencodes
cbs_70072ned_regiocodes
cbs_85237ned_measurecodes
cbs_85237ned_personenautos_brandstof
cbs_gebieden_2015
cbs_gebieden_2016
cbs_gebieden_2017
cbs_gebieden_2018
cbs_gebieden_2019
cbs_gebieden_2020
cbs_gebieden_2021
cbs_gebieden_2022
cbs_gebieden_2023
cbs_gebieden_2024
cbs_gebieden_2025
cbs_gebieden_2026
cbs_gebieden_measurecodes
dw_brandstof
dw_gebreken
dw_geconstateerde_gebreken
dw_gekentekende_voertuigen


In [74]:
def profiel(naam: str) -> pd.DataFrame:
    """Kolomprofiel van één bestand: type, uniciteit, NULL- en leeg-percentage."""
    samenvatting = q(f"summarize select * from {bron(naam)}")[
        ["column_name", "column_type", "min", "max", "approx_unique", "null_percentage"]
    ]

    # COLUMNS(*) past dezelfde expressie op elke kolom toe; UNPIVOT draait die ene
    # brede regel naar één regel per kolom, zodat het naast SUMMARIZE past.
    leeg = q(f"""
        unpivot (
            select count(*) filter (where trim(columns(*)) = '') from {bron(naam)}
        )
        on columns(*)
        into name column_name value n_leeg
    """)

    rijen = int(inventaris.loc[inventaris.bestand == f"{naam}.parquet", "rijen"].iloc[0])
    uit = samenvatting.merge(leeg, on="column_name", how="left")
    uit["null_percentage"] = uit["null_percentage"].astype(float)
    uit["leeg_percentage"] = (100 * uit["n_leeg"] / rijen).round(2)
    return uit.drop(columns=["n_leeg"])

tabel = profiel("rdw_gekentekende_voertuigen")

tabel

,column_name,column_type,min,max,approx_unique,null_percentage,leeg_percentage
0,kenteken,VARCHAR,0001TJ,10ZFXT,640083,0.00,0.0
1,voertuigsoort,VARCHAR,Aanhangwagen,Personenauto,14,0.00,0.0
2,merk,VARCHAR,3DOG CAMPING,ZWARTJES,1869,0.00,0.0
3,handelsbenaming,VARCHAR,250,toerist,27169,0.36,0.0
4,vervaldatum_apk,VARCHAR,19880128,20280920,5028,27.84,0.0
5,datum_tenaamstelling,VARCHAR,19651011,20260728,8296,7.04,0.0
6,bruto_bpm,VARCHAR,1,9999,22505,36.20,0.0
7,inrichting,VARCHAR,MPV,vuilniswagen,81,0.00,0.0
8,aantal_zitplaatsen,VARCHAR,1,93,67,20.98,0.0
9,eerste_kleur,VARCHAR,BEIGE,ZWART,16,0.00,0.0


## Grain

Voor elk bestand: is de veronderstelde sleutel echt uniek? `_sources.yml` claimt
"één rij per kenteken" en "één rij per kenteken per brandstofvolgnummer" — dat is
hier te toetsen in plaats van aan te nemen.

`geconstateerde_gebreken` heeft bewust géén sleutel: die is er niet, ook niet op
alle acht kolommen samen (zie §5).

In [75]:
KANDIDATEN = {
    "rdw_gekentekende_voertuigen": ["kenteken"],
    "rdw_brandstof": ["kenteken", "brandstof_volgnummer"],
    "rdw_gebreken": ["gebrek_identificatie"],
    "rdw_geconstateerde_gebreken": [
        "kenteken", "meld_datum_door_keuringsinstantie",
        "meld_tijd_door_keuringsinstantie", "gebrek_identificatie",
    ],
    "cbs_70072ned_motorvoertuigen_gemeente": ["RegioS", "Perioden", "Measure"],
    "cbs_85237ned_personenautos_brandstof": ["Measure", "Perioden", "Bouwjaar"],
    "cbs_70072ned_regiocodes": ["Identifier"],
    "cbs_70072ned_periodencodes": ["Identifier"],
    "cbs_70072ned_measurecodes": ["Identifier"],
    "cbs_85237ned_measurecodes": ["Identifier"],
    "cbs_gebieden_measurecodes": ["Identifier"],
}


def grain(naam: str, sleutel: list[str]) -> dict:
    kolommen = ", ".join(sleutel)
    uit = {"bestand": naam, "sleutel": " + ".join(sleutel)}
    try:
        r = q(f"""
            select
                count(*)                     as rijen,
                count(distinct ({kolommen})) as sleutels
            from {bron(naam)}
        """).iloc[0]
    except duckdb.Error as fout:
        # Meestal een kolomnaam die niet bestaat — dat is zelf een bevinding,
        # geen reden om de rest van de tabel niet te tonen.
        return uit | {"rijen": None, "sleutels": None, "uniek": None,
                      "opmerking": str(fout).splitlines()[0]}
    return uit | {
        "rijen": int(r.rijen),
        "sleutels": int(r.sleutels),
        "uniek": bool(r.rijen == r.sleutels),
        "rijen_per_sleutel": round(r.rijen / r.sleutels, 4),
    }


pd.DataFrame([grain(n, s) for n, s in KANDIDATEN.items()])

,bestand,sleutel,rijen,sleutels,uniek,rijen_per_sleutel
0,rdw_gekentekende_voertuigen,kenteken,500000,500000,True,1.0
1,rdw_brandstof,kenteken + brandstof_volgnummer,443167,443167,True,1.0
2,rdw_gebreken,gebrek_identificatie,1006,1006,True,1.0
3,rdw_geconstateerde_gebreken,kenteken + meld_datum_door_keuringsinstantie +...,1333714,1333713,False,1.0
4,cbs_70072ned_motorvoertuigen_gemeente,RegioS + Perioden + Measure,213101,213101,True,1.0
5,cbs_85237ned_personenautos_brandstof,Measure + Perioden + Bouwjaar,80592,80592,True,1.0
6,cbs_70072ned_regiocodes,Identifier,785,785,True,1.0
7,cbs_70072ned_periodencodes,Identifier,32,32,True,1.0
8,cbs_70072ned_measurecodes,Identifier,315,315,True,1.0
9,cbs_85237ned_measurecodes,Identifier,69,69,True,1.0


## Doubles

Een rij die op *alle* kolommen dubbel is, is nooit betekenisvol — het is ruis uit
de bron. `_sources.yml` noemt er één in `geconstateerde_gebreken`; hier wordt dat
over alle bestanden gecontroleerd.

In [84]:
def volledig_dubbel(naam: str) -> dict:
    r = q(f"""
        select
            (select count(*) from {bron(naam)})                       as rijen,
            (select count(*) from (select distinct * from {bron(naam)})) as uniek
    """).iloc[0]
    return {"bestand": naam, "rijen": int(r.rijen), "dubbel": int(r.rijen - r.uniek)}


namen = [p.stem for p in sorted(RAW.glob("*.parquet"))]

pd.DataFrame([volledig_dubbel(n) for n in namen])

,bestand,rijen,dubbel
0,cbs_70072ned_measurecodes,315,0
1,cbs_70072ned_motorvoertuigen_gemeente,213101,0
2,cbs_70072ned_periodencodes,32,0
3,cbs_70072ned_regiocodes,785,0
4,cbs_85237ned_measurecodes,69,0
5,cbs_85237ned_personenautos_brandstof,80592,0
6,cbs_gebieden_2015,22401,0
7,cbs_gebieden_2016,20670,0
8,cbs_gebieden_2017,19012,0
9,cbs_gebieden_2018,21660,0


In [87]:
# Print de dubbele rij
q(f"""
    select *, count(*) as n
    from {bron('rdw_geconstateerde_gebreken')}
    group by all
    having count(*) > 1
""")

,kenteken,soort_erkenning_keuringsinstantie,meld_datum_door_keuringsinstantie,meld_tijd_door_keuringsinstantie,gebrek_identificatie,soort_erkenning_omschrijving,aantal_gebreken_geconstateerd,meld_datum_door_keuringsinstantie_dt,n
0,02BND8,AZ,20260303,1639,205,APK Zware voertuigen,4,2026-03-03T16:39:00.000,2


## Aansluiting tussen de bestanden

De drie grote RDW-sets zijn op hetzelfde kentekenbereik opgehaald. Als dat klopt,
is elk kenteken in `brandstof` en `geconstateerde_gebreken` ook bekend in
`gekentekende_voertuigen` — anders zijn er feiten zonder dimensie.

In [103]:
def aansluiting(kind: str, kind_kolom: str, ouder: str, ouder_kolom: str) -> dict:
    r = q(f"""
        select
            count(*)                                            as rijen,
            count(distinct k.{kind_kolom})                      as sleutels,
            count(*) filter (where o.{ouder_kolom} is null)     as wees
        from {bron(kind)} k
        left join (select distinct {ouder_kolom} from {bron(ouder)}) o
               on k.{kind_kolom} = o.{ouder_kolom}
    """).iloc[0]
    return {
        "van": f"{kind}.{kind_kolom}",
        "naar": f"{ouder}.{ouder_kolom}",
        "rijen": int(r.rijen),
        "wees": int(r.wees),
        "dekking_pct": round(100 * (1 - r.wees / r.rijen), 2),
    }


pd.DataFrame([
    aansluiting("rdw_brandstof", "kenteken",
                "rdw_gekentekende_voertuigen", "kenteken"),
    aansluiting("rdw_geconstateerde_gebreken", "kenteken",
                "rdw_gekentekende_voertuigen", "kenteken"),
    aansluiting("rdw_geconstateerde_gebreken", "gebrek_identificatie",
                "rdw_gebreken", "gebrek_identificatie"),
    aansluiting("cbs_70072ned_motorvoertuigen_gemeente", "RegioS",
                "cbs_70072ned_regiocodes", "Identifier"),
    aansluiting("cbs_70072ned_motorvoertuigen_gemeente", "Measure",
                "cbs_70072ned_measurecodes", "Identifier"),
    aansluiting("cbs_70072ned_motorvoertuigen_gemeente", "Perioden",
                "cbs_70072ned_periodencodes", "Identifier"),
])

,van,naar,rijen,wees,dekking_pct
0,rdw_brandstof.kenteken,rdw_gekentekende_voertuigen.kenteken,443167,0,100.0
1,rdw_geconstateerde_gebreken.kenteken,rdw_gekentekende_voertuigen.kenteken,1333714,0,100.0
2,rdw_geconstateerde_gebreken.gebrek_identificatie,rdw_gebreken.gebrek_identificatie,1333714,0,100.0
3,cbs_70072ned_motorvoertuigen_gemeente.RegioS,cbs_70072ned_regiocodes.Identifier,213101,0,100.0
4,cbs_70072ned_motorvoertuigen_gemeente.Measure,cbs_70072ned_measurecodes.Identifier,213101,0,100.0
5,cbs_70072ned_motorvoertuigen_gemeente.Perioden,cbs_70072ned_periodencodes.Identifier,213101,0,100.0


## `cbs_gebieden_*`: twaalf bestanden, één relatie

Precies de constructie uit `_sources.yml`: het jaartal-patroon in de glob houdt
`cbs_gebieden_measurecodes.parquet` (ander schema) erbuiten, `filename=true`
levert het peiljaar.

In [105]:
GEBIEDEN = (
    f"read_parquet('{RAW}/cbs_gebieden_[0-9][0-9][0-9][0-9].parquet', "
    "filename = true, union_by_name = true)"
)

q(f"""
    select
        regexp_extract(filename, '(.*/)\\.parquet$', 1)                   as peiljaar,
        count(*)                                                          as rijen,
        count(distinct RegioS)                                            as regios,
        count(distinct RegioS) filter (where RegioS like 'GM%')           as gemeenten,
        count(distinct Measure)                                           as maten
    from {GEBIEDEN}
    group by 1
    order by 1
""")

,peiljaar,rijen,regios,gemeenten,maten
0,,234902,414,414,75


### Herindelingen

Het verschil in de gemeenteverzameling tussen twee opeenvolgende peiljaren is de
herindeling. Dat is de reden dat alle twaalf jaren binnengehaald zijn: `dim_gemeente`
moet weten wanneer een code verdween of ontstond.

In [124]:
q(f"""
    with per_jaar as (
        select distinct
            regexp_extract(filename, '(\\d{{4}})\\.parquet$', 1)::int as peiljaar,
            trim(RegioS)                                             as gemeente
        from {GEBIEDEN}
        where RegioS like 'GM%'
    ),
    verschil as (
        select
            peiljaar,
            gemeente,
            lag(peiljaar) over (partition by gemeente order by peiljaar)  as vorig,
            lead(peiljaar) over (partition by gemeente order by peiljaar) as volgend
        from per_jaar
    )
    select
        peiljaar,
        count(*) filter (where vorig is null   and peiljaar > (select min(peiljaar) from per_jaar)) as nieuw,
        count(*) filter (where volgend is null and peiljaar < (select max(peiljaar) from per_jaar)) as verdwenen
    from verschil
    group by 1
    having nieuw > 0 or verdwenen > 0
    order by 1
""")

,peiljaar,nieuw,verdwenen
0,2015,0,6
1,2016,3,3
2,2017,1,11
3,2018,3,34
4,2019,9,0
5,2020,0,4
6,2021,1,10
7,2022,3,4
8,2023,1,0


### `StringValue` is opgevuld

CBS levert `StringValue` met vaste breedte per maat. Zonder `trim()` mislukt elke
join op die kolom stilletjes — geen fout, gewoon nul rijen. Hieronder per maat het
verschil tussen ruwe en getrimde lengte.

In [125]:
q(f"""
    select
        Measure,
        count(*)                                          as rijen,
        max(length(StringValue))                          as max_lengte,
        max(length(trim(StringValue)))                    as max_lengte_getrimd,
        count(*) filter (where StringValue <> trim(StringValue)) as opgevuld
    from {GEBIEDEN}
    where StringValue is not null
    group by 1
    having opgevuld > 0
    order by rijen desc
    limit 15
""")

,Measure,rijen,max_lengte,max_lengte_getrimd,opgevuld
0,CS0001,4326,10,5,4326
1,PV0001,4326,10,4,4326
2,KK0001,4326,10,4,4326
3,ST0001,4326,10,1,4326
4,RT0001,4326,10,4,4326
5,VR0001,4326,10,4,4326
6,ST0002,4326,25,20,4326
7,VR0002,4326,50,25,4326
8,KK0002,4326,50,9,4326
9,JZ0001,4326,10,4,4326


## Toetsing van `stg_rdw__voertuigen`

Het staging-model doet harde `cast`s en vertaalt `'Ja'` naar een boolean. Beide
aannames zijn hier te controleren vóórdat `dbt build` erop stukloopt:

* een harde `cast` faalt op de eerste onparseerbare waarde;
* `kolom = 'Ja'` levert `false` bij élke andere waarde, ook bij `'N.v.t.'` — dat
  maakt van een driewaardig veld stilzwijgend een tweewaardig veld.

In [130]:
CASTS = {
    "aantal_zitplaatsen": "integer", "aantal_deuren": "integer",
    "aantal_wielen": "integer", "aantal_cilinders": "integer",
    "cilinderinhoud": "integer", "massa_ledig_voertuig": "integer",
    "massa_rijklaar": "integer", "toegestane_maximum_massa_voertuig": "integer",
    "maximale_constructiesnelheid": "integer", "catalogusprijs": "integer",
    "bruto_bpm": "integer",
    "datum_eerste_toelating_dt": "date", "datum_tenaamstelling_dt": "date",
    "vervaldatum_apk_dt": "date",
    "datum_eerste_tenaamstelling_in_nederland_dt": "date",
}

# try_cast geeft NULL waar cast zou crashen; het verschil met de al-NULL rijen is
# precies het aantal waarden dat dbt zou breken.
select = ",\n        ".join(
    f"count(*) filter (where {k} is not null and try_cast({k} as {t}) is null) as \"{k}\""
    for k, t in CASTS.items()
)

onparseerbaar = q(f"select {select} from {bron('rdw_gekentekende_voertuigen')}").T
onparseerbaar.columns = ["onparseerbaar"]
onparseerbaar["doeltype"] = pd.Series(CASTS)
onparseerbaar.sort_values("onparseerbaar", ascending=False)

,onparseerbaar,doeltype
aantal_zitplaatsen,0,integer
aantal_deuren,0,integer
aantal_wielen,0,integer
aantal_cilinders,0,integer
cilinderinhoud,0,integer
massa_ledig_voertuig,0,integer
massa_rijklaar,0,integer
toegestane_maximum_massa_voertuig,0,integer
maximale_constructiesnelheid,0,integer
catalogusprijs,0,integer


In [131]:
# Welke waarden komen echt voor in de kolommen die het model als boolean leest?
INDICATOREN = [
    "export_indicator", "tenaamstellen_mogelijk", "taxi_indicator",
    "openstaande_terugroepactie_indicator", "wam_verzekerd",
]

unions = "\nunion all\n".join(
    f"select '{k}' as kolom, coalesce({k}, '(null)') as waarde, count(*) as n "
    f"from {bron('rdw_gekentekende_voertuigen')} group by 2"
    for k in INDICATOREN
)

q(f"select * from ({unions}) order by kolom, n desc")

,kolom,waarde,n
0,export_indicator,Nee,465183
1,export_indicator,Ja,34817
2,openstaande_terugroepactie_indicator,Nee,469014
3,openstaande_terugroepactie_indicator,Ja,30986
4,taxi_indicator,Nee,499809
5,taxi_indicator,Ja,191
6,tenaamstellen_mogelijk,Ja,462254
7,tenaamstellen_mogelijk,Nee,37746
8,wam_verzekerd,Ja,372312
9,wam_verzekerd,N.v.t.,69436
